In [ ]:
print("Hello")

Hello


In [2]:
!jupyter kernelspec list


0.00s - Debugger warning: It seems that frozen modules are being used, which may
0.00s - make the debugger miss breakpoints. Please pass -Xfrozen_modules=off
0.00s - to python to disable frozen modules.
0.00s - Note: Debugging will proceed. Set PYDEVD_DISABLE_FILE_VALIDATION=1 to disable this validation.
Available kernels:
  julia      /root/.local/share/jupyter/kernels/julia
  ir         /usr/local/share/jupyter/kernels/ir
  python3    /usr/local/share/jupyter/kernels/python3


In [3]:
!pip install torch

In [4]:
!pip install -U -q transformers datasets bitsandbytes trl peft huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.4 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 645.5/645.5 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 16.5 MB/s eta 0:00:00:00:0100:01


In [6]:
from huggingface_hub import notebook_login
notebook_login()

Model

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"
config_8bit = BitsAndBytesConfig(load_in_8bit=True)
model_8bit = AutoModelForCausalLM.from_pretrained(model_name,
                                                  quantization_config=config_8bit,
                                                  device_map="auto",
                                                  dtype=torch.float16,
                                                  trust_remote_code=True,
                                                  )


config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Tokenizer

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_name, padding_side="left", trust_remote_code="True")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Dataset

In [9]:
from datasets import load_dataset
dataset = load_dataset("MattCoddity/dockerNLcommands")
dataset

README.md: 0.00B [00:00, ?B/s]

06102023.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2415 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 2415
    })
})

In [10]:
dataset["train"][0]

{'input': 'Give me a list of containers that have the Ubuntu image as their ancestor.',
 'output': "docker ps --filter 'ancestor=ubuntu'",
 'instruction': 'translate this sentence in docker command'}

In [11]:
from datasets import DatasetDict

train_test_split = dataset['train'].train_test_split(test_size=0.2
                                                     )
dataset = DatasetDict({
    'train': train_test_split['train'],
    'validation': train_test_split['test']
})
dataset

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction'],
        num_rows: 483
    })
})

In [12]:
def to_chat_template(example):

    messages = [
        {"role": "system", "content": example['instruction']},
        {"role": "user", "content": example['input']},
        {"role": "assistant", "content": example['output']}
    ]

    
    return {'text':messages}

dataset = dataset.map(to_chat_template)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [13]:
dataset['train'][0]

{'input': 'I am looking for the images created since the nginx:alpine image.',
 'output': 'docker images -f since=nginx:alpine',
 'instruction': 'translate this sentence in docker command',
 'text': [{'content': 'translate this sentence in docker command',
   'role': 'system'},
  {'content': 'I am looking for the images created since the nginx:alpine image.',
   'role': 'user'},
  {'content': 'docker images -f since=nginx:alpine', 'role': 'assistant'}]}

In [14]:
tokenizer.mistral = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.1")
tokenizer.mistral.apply_chat_template(dataset['train']['text'][0], tokenize=False)

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

'<s> [INST] translate this sentence in docker command\n\nI am looking for the images created since the nginx:alpine image. [/INST] docker images -f since=nginx:alpine</s>'

In [15]:
tokenizer.apply_chat_template(dataset['train']['text'][0],tokenize=False)

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 19 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI am looking for the images created since the nginx:alpine image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f since=nginx:alpine<|eot_id|>'

In [16]:
def apply_chat_template(example):

    new_text = tokenizer.apply_chat_template(example['text'],tokenize=False)
    return {'text': new_text}

dataset = dataset.map(apply_chat_template)
dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input', 'output', 'instruction', 'text'],
        num_rows: 483
    })
})

In [17]:
dataset['train']['text'][0]

'<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 19 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nI am looking for the images created since the nginx:alpine image.<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\ndocker images -f since=nginx:alpine<|eot_id|>'

In [18]:
tokenizer(dataset['train']['text'][0])

{'input_ids': [128000, 128000, 128006, 9125, 128007, 271, 38766, 1303, 33025, 2696, 25, 6790, 220, 2366, 18, 198, 15724, 2696, 25, 220, 777, 5186, 220, 2366, 21, 271, 14372, 420, 11914, 304, 27686, 3290, 128009, 128006, 882, 128007, 271, 40, 1097, 3411, 369, 279, 5448, 3549, 2533, 279, 71582, 25, 278, 39138, 2217, 13, 128009, 128006, 78191, 128007, 271, 29748, 5448, 482, 69, 2533, 28, 74661, 25, 278, 39138, 128009], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [19]:
def tokenize_fn(example): 
    return tokenizer(example['text'])

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=['input', 'output', 'instruction', 'text'])
tokenized_dataset

Map:   0%|          | 0/1932 [00:00<?, ? examples/s]

Map:   0%|          | 0/483 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [20]:
len(tokenized_dataset['train']['input_ids'][1])

94

In [21]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False, return_tensors = "pt")

In [22]:
tokenizer.pad_token

In [23]:
tokenizer.pad_token = tokenizer.eos_token
tokenizer.pad_token

'<|eot_id|>'

#Data Loader

In [24]:

from torch.utils.data import DataLoader

data_loader = DataLoader(
    tokenized_dataset['train'],
    batch_size = 2,
    collate_fn = data_collator
)

for step, batch in enumerate(data_loader):
    print(f"Batch {step}")
    print("input_ids.shape: ", batch['input_ids'].shape)

    if step >= 3: 
        break

Batch 0
input_ids.shape:  torch.Size([2, 94])
Batch 1
input_ids.shape:  torch.Size([2, 78])
Batch 2
input_ids.shape:  torch.Size([2, 71])
Batch 3
input_ids.shape:  torch.Size([2, 68])


LoRa

In [25]:
from peft import LoraConfig, get_peft_model
import copy

model_8bit_clone = copy.deepcopy(model_8bit)

lora_config = LoraConfig(

    r = 32,
    lora_alpha = 16, #the higher value will be , lora matrix will get influence than matrix of base model
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "down_proj", "up_proj"],
    lora_dropout = 0.05,
    bias = "none", 
    task_type = "CAUSAL_LM"  
)

model_8bit_lora = get_peft_model(model_8bit_clone, lora_config)
model_8bit_lora.print_trainable_parameters()

trainable params: 22,544,384 || all params: 1,258,358,784 || trainable%: 1.7916


In [26]:
model_8bit

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear8bitLt(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear8bitLt(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear8bitLt(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm)

Hyper Traning

In [27]:

from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
                                    output_dir = "./results",
                                    per_device_train_batch_size = 2,
                                    per_device_eval_batch_size = 2,
                                    eval_steps = 10,
                                    eval_strategy = "steps",
                                    save_steps = 10,
                                    save_strategy = "steps",
                                    # num_train_epochs = 1,
                                    max_steps = 60,
                                    logging_steps = 10,
                                    learning_rate = 3e-5,
                                    weight_decay = 0.01,
                                    warmup_steps = 0,
                                    # fp16 = False,
                                    # bf16 = True,
                                    gradient_accumulation_steps = 4,
                                    optim = "adamw_torch",
                                    report_to = "none"
)


Training

In [28]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1932
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 483
    })
})

In [29]:

trainer = SFTTrainer(
    model = model_8bit_lora,
    train_dataset = tokenized_dataset['train'],
    eval_dataset = tokenized_dataset['validation'],
    data_collator = data_collator,
    # tokenizer = tokenizer,
    args = training_args,
)

trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009}.


Step,Training Loss,Validation Loss
10,4.175104,3.516656
20,3.148163,2.814739
30,2.534925,2.325334
40,2.174214,2.056669
50,1.934368,1.924495
60,1.883143,1.888336


TrainOutput(global_step=60, training_loss=2.6416527112325032, metrics={'train_runtime': 308.9333, 'train_samples_per_second': 1.554, 'train_steps_per_second': 0.194, 'total_flos': 225440259342336.0, 'train_loss': 2.6416527112325032})

In [30]:
model_8bit_lora

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear8bitLt(
                (base_layer): Linear8bitLt(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj):

LoRa with Trained Model 

In [31]:
import torch 
torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

model_8bit_weights = layer0.self_attn.q_proj.weight

print("mddel_8bit_weights ", model_8bit_weights)

mddel_8bit_weights  Parameter containing:
Parameter(Int8Params([[-44,  16,  61,  ..., -22, -29,  50],
            [ 14,  70,  65,  ..., -39, -18,  13],
            [ 16,  14,  30,  ..., -34, -34, -24],
            ...,
            [ 17,  20,  40,  ..., -40, -15, -16],
            [ 32, -35,  50,  ..., -17, -41, -21],
            [-14, -30,  -7,  ...,  30,   5,  -2]], device='cuda:0',
           dtype=torch.int8))


In [32]:
import torch

torch.set_printoptions(precision=8, sci_mode=False)

layer0 = model_8bit_lora.model.model.layers[0]

weight_A = layer0.self_attn.q_proj.lora_A["default"].weight

print("LoRA A weights ", weight_A)

LoRA A weights  Parameter containing:
tensor([[-0.00982666,  0.01196289, -0.02050781,  ...,  0.00001764,
         -0.01745605, -0.01281738],
        [ 0.01831055, -0.01721191, -0.00116730,  ...,  0.01263428,
          0.00781250,  0.00729370],
        [ 0.02209473,  0.01953125,  0.00335693,  ..., -0.00842285,
         -0.00034523,  0.00473022],
        ...,
        [-0.00331116, -0.02026367, -0.02026367,  ...,  0.01116943,
          0.02136230,  0.01733398],
        [ 0.01092529,  0.02172852,  0.00168610,  ..., -0.00848389,
          0.00361633, -0.00842285],
        [ 0.01452637, -0.00268555, -0.01684570,  ..., -0.00165558,
         -0.00352478,  0.01031494]], device='cuda:0', dtype=torch.bfloat16,
       requires_grad=True)


Base Model

In [33]:
base_model = AutoModelForCausalLM.from_pretrained(
                                                   
                                                   model_name,
                                                   device_map="auto",
                                                   trust_remote_code=True
                                                   
)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

In [34]:
base_model 

#base_model is the orginal version of dataset

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

In [35]:
import os

lora_path = "./lora_adapters"
model_8bit_lora.save_pretrained(lora_path)

In [37]:
!pip install -U torchao
from peft import PeftModel

base_model = PeftModel.from_pretrained(base_model, lora_path)
base_model

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.6 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 2048)
        (layers): ModuleList(
          (0-15): 16 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=2048, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=2048, out_features=32, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=32, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

Fully Implentation of LoRA

In [38]:
base_model = base_model.merge_and_unload()
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (ro

Uploading model to huggingface


In [39]:
base_model.push_to_hub("yahya2004/Llama3.2-Docker")
tokenizer.push_to_hub("yahya2004/Llama3.2-Docker")

README.md: 0.00B [00:00, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...qjagn6r/model.safetensors:   6%|6         |  160MB / 2.47GB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpo1qrjfgg/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/yahya2004/Llama3.2-Docker/commit/80798c278d38ae29cbb32c0a9a60fc304ba2e499', commit_message='Upload tokenizer', commit_description='', oid='80798c278d38ae29cbb32c0a9a60fc304ba2e499', pr_url=None, repo_url=RepoUrl('https://huggingface.co/yahya2004/Llama3.2-Docker', endpoint='https://huggingface.co', repo_type='model', repo_id='yahya2004/Llama3.2-Docker'), pr_revision=None, pr_num=None)

LLM

In [40]:
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=base_model,
    dtype=torch.bfloat16,
    device_map="auto",
    tokenizer=tokenizer
)
messages = [
    {"role": "system", "content": "Translate this sentence in Docker command"},
    {"role": "user", "content": "please show me the Docker containers that have exited and related to the mongo image"},
]
outputs = pipe(
    messages,
    max_new_tokens=256,
    temperature = 0.5,
    top_p = 0.9,
    top_k = 10,
    do_sample = True,
    repetition_penalty = 1.2
)
print(outputs[0]["generated_text"][-1]['content'])


Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature', 'repetition_penalty', 'top_k', 'top_p'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


To get a list of all running containers, including those associated with MongoDB (based on your previous question), you can use the following `docker ps` command:

```bash
docker ps -a --filter "image=postgres" --format "%s"
```

This will display a list of all containers currently running. The filter `-f "image=postgres"` ensures we only see containers tagged as 'postgres', which is typically used for PostgreSQL databases.

However, if you're looking specifically at containers associated with the MongoDB image (`mongo`) or its variants like MongoDB Server, you might want to refine your search by adding additional filters:

- For the official MongoDB image:
    ```bash
docker ps -aq --filter "image=mongodb/mongod"
```
    
- If using an older version of MongoDB, consider the following options instead.
    ```
    docker ps -qo --filter "image=postgres"
    ```

The last one (`--qo`) sorts the output alphabetically before filtering out non-matching images. This approach helps prioritize

In [41]:
dataset['train'][1]

{'input': 'Docker, I request you to log in to the registry yetanotherregistry.example.net as user "marydoe" with the password "herpassword".',
 'output': '"docker login yetanotherregistry.example.net --username=marydoe --password=herpassword"',
 'instruction': 'translate this sentence in docker command',
 'text': '<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 19 Apr 2026\n\ntranslate this sentence in docker command<|eot_id|><|start_header_id|>user<|end_header_id|>\n\nDocker, I request you to log in to the registry yetanotherregistry.example.net as user "marydoe" with the password "herpassword".<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"docker login yetanotherregistry.example.net --username=marydoe --password=herpassword"<|eot_id|>'}